In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import zipfile
import os

zip_path = r"/content/drive/MyDrive/Merged Dataset.zip"  # adjust name
extract_path = "/content/drive/MyDrive"

try:
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ Dataset extracted into Drive")
except (ConnectionAbortedError, OSError) as e:
    print(f"❌ Error during extraction: {e}")
    print("It appears the connection to Google Drive was interrupted.")
    print("Please ensure your Google Drive is properly mounted and try re-running this cell.")
    print("You might need to re-mount Google Drive using 'from google.colab import drive; drive.mount('/content/drive')' in a previous cell if the issue persists.")

OSError: [Errno 107] Transport endpoint is not connected

In [ ]:
import os

dataset_path = r"/content/drive/MyDrive/Merged Dataset"
print(os.listdir(dataset_path))

['buffalo', 'cow']


In [ ]:
cow_count = len(os.listdir(r'/content/drive/MyDrive/Merged Dataset/cow'))
buffalo_count = len(os.listdir(r'/content/drive/MyDrive/Merged Dataset/buffalo'))

print("Cow images:", cow_count)
print("Buffalo images:", buffalo_count)

Cow images: 1386
Buffalo images: 900


In [ ]:
import tensorflow as tf

dataset_path = r"/content/drive/MyDrive/Merged Dataset"

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

Found 2286 files belonging to 2 classes.
Using 1829 files for training.
Found 2286 files belonging to 2 classes.
Using 457 files for validation.


In [ ]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

base_model = EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False  # freeze pretrained weights

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(2, activation='softmax')
])

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 832s 14s/step - accuracy: 0.5703 - loss: 0.6972 - val_accuracy: 0.6061 - val_loss: 0.6748
Epoch 2/10
 2/58 ━━━━━━━━━━━━━━━━━━━━ 4s 77ms/step - accuracy: 0.5078 - loss: 0.7472 

KeyboardInterrupt: 

In [ ]:
base_model.trainable = True

# Freeze early layers, train only deeper ones
for layer in base_model.layers[:-20]:
    layer.trainable = False

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

REBULDING MODEL AGAIN PROPERLY WITH AUGMENTATION BECAUSE OF FACING "**FEATURE SEPARABILITY PROBLEM**" IN THE UPPER MODULE 👇👇👇

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

In [ ]:
model = models.Sequential([
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(2, activation='softmax')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_aug = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 26s 206ms/step - accuracy: 0.5890 - loss: 0.6848 - val_accuracy: 0.6061 - val_loss: 0.6762
Epoch 2/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 11s 187ms/step - accuracy: 0.6172 - loss: 0.6669 - val_accuracy: 0.6061 - val_loss: 0.6705
Epoch 3/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 10s 169ms/step - accuracy: 0.6054 - loss: 0.6743 - val_accuracy: 0.6061 - val_loss: 0.6705
Epoch 4/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 9s 150ms/step - accuracy: 0.6062 - loss: 0.6770 - val_accuracy: 0.6061 - val_loss: 0.6705
Epoch 5/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 10s 168ms/step - accuracy: 0.6046 - loss: 0.6765 - val_accuracy: 0.6061 - val_loss: 0.6710
Epoch 6/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 10s 171ms/step - accuracy: 0.6154 - loss: 0.6720 - val_accuracy: 0.6061 - val_loss: 0.6705
Epoch 7/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 9s 155ms/step - accuracy: 0.6196 - loss: 0.6704 - val_accuracy: 0.6061 - val_loss: 0.6705
Epoch 8/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 10s 171ms/step - accuracy: 0.6138 - loss: 0.6693 - val_accura

SINCE VAL_ACCURACY IS STILL CONSTANT = 0.6061 EVEN AFTER AUGMENTATION, GONNA USE A DIFFERENT MODEL TO TEST FOR BETTER OUTPUT.
NEW MODEL :--- **MobileNetV2**

In [ ]:
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [ ]:
model_mobilenet = tf.keras.Sequential([
    data_augmentation,  # reuse augmentation layer
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(2, activation='softmax')
])

In [ ]:
model_mobilenet.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_mobilenet = model_mobilenet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 16s 188ms/step - accuracy: 0.5392 - loss: 0.8426 - val_accuracy: 0.6061 - val_loss: 0.6713
Epoch 2/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 10s 169ms/step - accuracy: 0.6076 - loss: 0.6805 - val_accuracy: 0.6061 - val_loss: 0.6747
Epoch 3/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 10s 172ms/step - accuracy: 0.6181 - loss: 0.6727 - val_accuracy: 0.6061 - val_loss: 0.6704
Epoch 4/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 9s 151ms/step - accuracy: 0.6166 - loss: 0.6646 - val_accuracy: 0.6061 - val_loss: 0.6717
Epoch 5/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 10s 168ms/step - accuracy: 0.6034 - loss: 0.6750 - val_accuracy: 0.6061 - val_loss: 0.6710
Epoch 6/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 10s 167ms/step - accuracy: 0.6061 - loss: 0.6807 - val_accuracy: 0.6061 - val_loss: 0.6706
Epoch 7/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 9s 147ms/step - accuracy: 0.6151 - loss: 0.6761 - val_accuracy: 0.6061 - val_loss: 0.6709
Epoch 8/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 10s 170ms/step - accuracy: 0.6142 - loss: 0.6728 - val_accura

USING QWEN3 LLM --- DOING EVERYTHING FROM SCRATCH

WHOLE NEW APPROACH ✅

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

print(os.listdir('/content/drive/MyDrive'))

['Google AI Studio', 'To-do list.gsheet', 'Colab Notebooks', 'Great! now please tell me how to bring best out o....gsheet', 'NoteBook LM PROMPT HELP.gsheet', 'Google Earth', 'Merged Dataset.zip', 'Merged Dataset']


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import numpy as np
import os

# Configure GPU memory growth (prevents OOM errors)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = r"/content/drive/MyDrive/Merged Dataset"

# Load datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'  # Crucial for sparse_categorical_crossentropy
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

# DEBUG: Check Class Distribution (moved before shuffle/prefetch)
class_names = train_ds.class_names
print(f"Classes: {class_names}")
print(f"Number of classes: {len(class_names)}")

# OPTIMIZE PERFORMANCE
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)


# Check if one class dominates (Imbalance Check)
for i, batch in enumerate(val_ds.take(1)):
    labels = batch[1].numpy()
    unique, counts = np.unique(labels, return_counts=True)
    print(f"Validation Class Distribution: {dict(zip(unique, counts))}")

Found 2286 files belonging to 2 classes.
Using 1829 files for training.
Found 2286 files belonging to 2 classes.
Using 457 files for validation.
Classes: ['buffalo', 'cow']
Number of classes: 2
Validation Class Distribution: {np.int32(0): np.int64(11), np.int32(1): np.int64(21)}


In [ ]:
NUM_CLASSES = len(class_names)

# 1. Data Augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="augmentation")

# 2. Input Layer
inputs = tf.keras.Input(shape=(224, 224, 3))

# 3. Augmentation
x = data_augmentation(inputs)

# 4. CRITICAL: Preprocessing for MobileNetV2
x = preprocess_input(x)

# 5. Base Model
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze base

# 6. Classification Head
x = base_model(x, training=False) # Set training=False for BatchNorm layers
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# DEBUG: Check Trainable Variables
print(f"\nTotal Layers: {len(model.layers)}")
print(f"Trainable Variables: {len(model.trainable_variables)}")
if len(model.trainable_variables) == 0:
    print("⚠️ WARNING: No trainable variables! Model will not learn.")
else:
    print("✅ Model has trainable layers.")


Total Layers: 6
Trainable Variables: 2
✅ Model has trainable layers.


In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        'best_model.h5', save_best_only=True, monitor='val_accuracy', mode='max'
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=3, restore_best_weights=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)

Epoch 1/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.7123 - loss: 0.5725

58/58 ━━━━━━━━━━━━━━━━━━━━ 309s 1s/step - accuracy: 0.8229 - loss: 0.3915 - val_accuracy: 0.9322 - val_loss: 0.1953
Epoch 2/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.9185 - loss: 0.1956

58/58 ━━━━━━━━━━━━━━━━━━━━ 12s 87ms/step - accuracy: 0.9191 - loss: 0.2019 - val_accuracy: 0.9497 - val_loss: 0.1541
Epoch 3/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.9360 - loss: 0.1561

58/58 ━━━━━━━━━━━━━━━━━━━━ 12s 88ms/step - accuracy: 0.9328 - loss: 0.1699 - val_accuracy: 0.9519 - val_loss: 0.1356
Epoch 4/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.9334 - loss: 0.1537

58/58 ━━━━━━━━━━━━━━━━━━━━ 11s 91ms/step - accuracy: 0.9415 - loss: 0.1448 - val_accuracy: 0.9562 - val_loss: 0.1230
Epoch 5/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.9429 - loss: 0.1428

58/58 ━━━━━━━━━━━━━━━━━━━━ 12s 102ms/step - accuracy: 0.9459 - loss: 0.1437 - val_accuracy: 0.9628 - val_loss: 0.1153
Epoch 6/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 11s 86ms/step - accuracy: 0.9590 - loss: 0.1101 - val_accuracy: 0.9606 - val_loss: 0.1156
Epoch 7/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 11s 82ms/step - accuracy: 0.9601 - loss: 0.1096 - val_accuracy: 0.9628 - val_loss: 0.1039
Epoch 8/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 11s 82ms/step - accuracy: 0.9606 - loss: 0.1104 - val_accuracy: 0.9628 - val_loss: 0.1047
Epoch 9/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.9583 - loss: 0.1038

58/58 ━━━━━━━━━━━━━━━━━━━━ 12s 108ms/step - accuracy: 0.9563 - loss: 0.1154 - val_accuracy: 0.9694 - val_loss: 0.0952
Epoch 10/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.9673 - loss: 0.0898

58/58 ━━━━━━━━━━━━━━━━━━━━ 12s 107ms/step - accuracy: 0.9650 - loss: 0.0937 - val_accuracy: 0.9759 - val_loss: 0.0914
Epoch 11/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 11s 91ms/step - accuracy: 0.9595 - loss: 0.0978 - val_accuracy: 0.9672 - val_loss: 0.0911
Epoch 12/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 11s 84ms/step - accuracy: 0.9688 - loss: 0.0809 - val_accuracy: 0.9694 - val_loss: 0.0875
Epoch 13/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 12s 83ms/step - accuracy: 0.9666 - loss: 0.0862 - val_accuracy: 0.9716 - val_loss: 0.0822
Epoch 14/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 11s 86ms/step - accuracy: 0.9694 - loss: 0.0855 - val_accuracy: 0.9759 - val_loss: 0.0779
Epoch 15/15
58/58 ━━━━━━━━━━━━━━━━━━━━ 12s 102ms/step - accuracy: 0.9727 - loss: 0.0795 - val_accuracy: 0.9737 - val_loss: 0.0757


In [ ]:
import os
os.listdir()

['.config',
 'bovision_ai_classifier.h5',
 'drive',
 'bovision_ai_classifier_fixed.h5',
 'sample_data']

In [ ]:
model = tf.keras.models.load_model("bovision_ai_classifier.h5")
model.save("bovision_ai_classifier_fixed.h5")

ValueError: Unknown layer: 'TrueDivide'. Please ensure you are using a `keras.utils.custom_object_scope` and that this object is included in the scope. See https://www.tensorflow.org/guide/keras/save_and_serialize#registering_the_custom_object for details.

In [ ]:
# Update your callback to use .keras instead of .h5
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        'best_model.keras',  # <-- Changed extension
        save_best_only=True,
        monitor='val_accuracy',
        mode='max'
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,  # Increased patience slightly
        restore_best_weights=True
    )
]

NameError: name 'tf' is not defined

In [ ]:
model.save('bovision_ai_classifier.keras')
print("✅ Model saved successfully!")

✅ Model saved successfully!


In [ ]:
# Download the model to your local machine
from google.colab import files
files.download('bovision_ai_classifier.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import tensorflow as tf

# Load the saved model (this code would run on your local machine or web server)
loaded_model = tf.keras.models.load_model('bovision_ai_classifier.keras')

loaded_model.summary()
print("\n✅ Model loaded successfully from local storage!")

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 4 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │         2,562 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,263,110 (8.63 MB)

 Trainable params: 2,562 (10.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

 Optimizer params: 2,564 (10.02 KB)


✅ Model loaded successfully from local storage!


In [ ]:
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
  print(f'User uploaded file "{filename}" with length {len(uploaded[filename])} bytes')

# You can then access the uploaded file in the /content/ directory
# For example, to list the files in /content/:
# import os
# print(os.listdir('/content/'))

Saving test_img.jpg to test_img.jpg
User uploaded file "test_img.jpg" with length 90606 bytes


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Assuming you have an image file, e.g., 'test_image.jpg'
# For demonstration, let's create a dummy image (replace with actual image loading)
# In a real scenario, you'd load an image uploaded by the user.

# Create a dummy image for demonstration purposes
dummy_image_path = '/content/test_img.jpg'
img_size = (224, 224)

# Let's try to grab an image from our dataset to simulate a prediction
for images, labels in val_ds.take(1):
    sample_image = images[0].numpy() # Get the first image from the batch
    sample_label = labels[0].numpy() # Get its label

    # The image from val_ds is already preprocessed (normalized 0-1) and resized
    # For a new, raw image, you would do:
    # img = image.load_img(image_path, target_size=img_size)
    # img_array = image.img_to_array(img)
    # img_array = np.expand_dims(img_array, axis=0) # Create a batch dimension
    # img_array = preprocess_input(img_array) # MobileNetV2 specific preprocessing

    # For our sample_image from val_ds, it's already in the right format for the model
    # except for the batch dimension and MobileNetV2 preprocessing needs.
    # The model expects images in the range of -1 to 1 after preprocess_input.
    # Since our `train_ds` and `val_ds` already include `preprocess_input`, we can use `sample_image` directly.

    # We need to ensure the image has the correct preprocessing.
    # If the loaded_model already includes the `preprocess_input` layer (which it does via the `model` definition in Gf89zK0SAVUs):
    input_image_for_prediction = np.expand_dims(sample_image, axis=0) # Add batch dimension

    predictions = loaded_model.predict(input_image_for_prediction)
    predicted_class_index = np.argmax(predictions[0])
    predicted_class_name = class_names[predicted_class_index]

    true_class_name = class_names[sample_label]

    print(f"Predicted class: {predicted_class_name} (Probability: {np.max(predictions[0]):.2f})")
    print(f"True class: {true_class_name}")
    break # Just take one image for example

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Predicted class: cow (Probability: 0.97)
True class: cow
